# Kuggle 8주차 과제 · CNN으로 이미지 분류하기

**목표** — 손글씨 숫자 이미지를 분류하는 작은 CNN을 완성하고, 실제 예측 결과로 정확도와 오분류를 해석합니다.

1. Google Colab에서 이 노트북을 엽니다. 기본 CPU 런타임으로 실행할 수 있습니다.
2. 위에서부터 실행하면서 `TODO 1~3`의 `None`을 정수로 바꾸세요. 이미지 데이터는 scikit-learn에 내장되어 있습니다.
3. 마지막에 **단답형 3문항**을 작성하고, 그래프와 실행 결과가 보이는 `.ipynb`를 GitHub에 올리세요.

평가 기준은 **빈칸 완성, 코드 실행, 단답형 답안**입니다. 특정 정확도를 맞힐 필요는 없습니다.

## 01. 이미지와 픽셀 · 손글씨 숫자 살펴보기

숫자 0-9를 손으로 쓴 8x8 회색조 이미지 1,797장을 사용합니다. 각 이미지는 채널이 하나인 **1 x 8 x 8** 배열입니다. 밝기 값(0-16)은 16으로 나눠 0-1 범위로 바꿉니다.

데이터를 섞어 **학습 1,200장 / 검증 300장 / 테스트 297장**으로 나눕니다. `DataLoader`는 여러 이미지를 묶어 배치 단위로 모델에 전달합니다.

In [ ]:
import copy

import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, TensorDataset, random_split

SEED = 777
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 0.001

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    torch.set_num_threads(min(4, torch.get_num_threads()))
print('Device:', device)

In [ ]:
digits = load_digits()  # scikit-learn 패키지에 포함된 이미지
images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
labels = torch.tensor(digits.target, dtype=torch.long)
dataset = TensorDataset(images, labels)

train_data, val_data, test_data = random_split(
    dataset, [1200, 300, 297], generator=torch.Generator().manual_seed(SEED)
)
class_names = [str(number) for number in range(10)]

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

sample_image, sample_label = train_data[0]
print('Image shape (C, H, W):', tuple(sample_image.shape))
print('Train / validation / test:', len(train_data), '/', len(val_data), '/', len(test_data))
plt.imshow(sample_image[0], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
plt.title(class_names[sample_label.item()])
plt.axis('off')
plt.show()

## 02. CNN의 특징 추출 · 합성곱과 맥스 풀링

합성곱 → ReLU → 맥스 풀링을 한 블록으로 묶어 두 번 적용합니다. 아래 표는 **이미지 한 장** 기준의 크기입니다.

| 단계 | 출력 형태 |
|---|---|
| 회색조 입력 | 1 × 8 × 8 |
| 첫 블록의 합성곱 | 16 × 8 × 8 |
| 첫 블록의 풀링 | 16 × 4 × 4 |
| 두 번째 블록의 합성곱 | 32 × 4 × 4 |
| 두 번째 블록의 풀링 | 32 × 2 × 2 |
| Flatten | 길이 = 채널 수 × 높이 × 너비 |
| 완전연결층 | 클래스별 점수 10개 |

- **TODO 1** — 회색조 입력의 채널 수. 위 표에서 입력의 첫 번째 숫자를 확인하세요.
- **TODO 2** — 2×2 풀링 창이 겹치지 않게 한 번에 이동하는 칸 수. 창의 한 변 길이만큼 이동합니다.
- **TODO 3** — 두 번째 풀링 출력의 채널 × 높이 × 너비를 곱한 값.

In [ ]:
INPUT_CHANNELS = ___  # TODO 1
POOL_STRIDE = ___      # TODO 2
FLAT_FEATURES = ___    # TODO 3
PADDING = 1             # 합성곱 전후 가로·세로 크기 유지

settings = (INPUT_CHANNELS, POOL_STRIDE, FLAT_FEATURES)
if any(value is None for value in settings):
    raise ValueError('이 셀의 TODO 1~3을 모두 채운 뒤 다시 실행하세요.')

block1 = nn.Sequential(
    nn.Conv2d(INPUT_CHANNELS, 16, kernel_size=3, stride=1, padding=PADDING),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=POOL_STRIDE),
)
block2 = nn.Sequential(
    nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=PADDING),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=POOL_STRIDE),
)
model = nn.Sequential(
    block1,
    block2,
    nn.Flatten(start_dim=1),  # 배치 축은 유지
    nn.Linear(FLAT_FEATURES, len(class_names)),
).to(device)

# 실제 이미지 한 장으로 학습 전 출력 크기 확인
with torch.no_grad():
    sample_batch = sample_image.unsqueeze(0).to(device)
    features1 = block1(sample_batch)
    features2 = block2(features1)
    flat = model[2](features2)
    print('Block 1:', tuple(features1.shape))
    print('Block 2:', tuple(features2.shape))
    print('Flatten:', tuple(flat.shape))
    if flat.shape[1] != FLAT_FEATURES:
        raise ValueError('Flatten 길이를 확인하고 TODO 1~3을 다시 살펴보세요.')
    print('Logits:', tuple(model[3](flat).shape))

## 03. 분류와 학습 · 점수로 손실을 계산하고 가중치 갱신

완전연결층은 이미지별 클래스 점수(**logits**)를 출력합니다.
`CrossEntropyLoss`에 logits와 정답 클래스 번호를 전달합니다. Softmax 확률은 예측 예시를 읽을 때 사용합니다.

이미지마다 점수가 가장 높은 클래스를 예측으로 선택합니다.

In [ ]:
def predict_classes(logits):
    return logits.argmax(dim=1)

loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += loss_fn(logits, labels).item() * len(labels)
        correct += (predict_classes(logits) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total

In [ ]:
# 다시 학습하려면 02절 모델 정의 셀부터 실행하세요.
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()               # 이전 배치의 기울기 초기화
        logits = model(images)              # 순전파
        loss = loss_fn(logits, labels)       # 손실 계산
        loss.backward()                     # 역전파
        optimizer.step()                    # 필터와 분류기의 가중치 갱신
        running_loss += loss.item() * len(labels)

    train_loss = running_loss / len(train_data)
    val_loss, val_acc = evaluate(model, val_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_weights = copy.deepcopy(model.state_dict())
    print(f'Epoch {epoch:02d} | train loss {train_loss:.3f} | '
          f'val loss {val_loss:.3f} | val accuracy {val_acc:.1%}')

model.load_state_dict(best_weights)
print(f'검증 손실이 가장 낮은 모델: epoch {best_epoch}')

In [ ]:
epochs = range(1, EPOCHS + 1)
plt.plot(epochs, train_losses, marker='o', label='Train loss')
plt.plot(epochs, val_losses, marker='o', label='Validation loss')
plt.axvline(best_epoch, color='gray', linestyle='--', label='Selected epoch')
plt.xticks(list(epochs))
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.legend()
plt.tight_layout()
plt.show()

## 04. 모델 평가 · 정확도, 혼동행렬, 오분류

검증 손실로 선택한 모델을 따로 떼어 둔 테스트 데이터에서 평가합니다. 혼동행렬의 **행은 실제 클래스, 열은 예측 클래스**입니다. 아래의 사진과 예측값은 이번 실행에서 얻은 결과입니다.

In [ ]:
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

true_batches, pred_batches, confidence_batches = [], [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        predicted = predict_classes(logits)
        probabilities = logits.softmax(dim=1)
        confidence = probabilities.gather(1, predicted.unsqueeze(1)).squeeze(1)
        true_batches.append(labels)
        pred_batches.append(predicted.cpu())
        confidence_batches.append(confidence.cpu())

true_labels = torch.cat(true_batches)
pred_labels = torch.cat(pred_batches)
confidences = torch.cat(confidence_batches)
num_classes = len(class_names)

# (실제, 예측) 쌍을 행 우선 순서로 세어 10 × 10 표 구성
pair_ids = num_classes * true_labels + pred_labels
cm = torch.bincount(pair_ids, minlength=num_classes ** 2).reshape(num_classes, num_classes)
accuracy = cm.diag().sum().item() / cm.sum().item()
print(f'Test accuracy: {accuracy:.1%} ({cm.diag().sum().item()} / {len(test_data)})')

fig, ax = plt.subplots(figsize=(9, 7))
heatmap = ax.imshow(cm.numpy(), cmap='Blues')
ax.set_xticks(range(num_classes), class_names, rotation=45, ha='right')
ax.set_yticks(range(num_classes), class_names)
ax.set(xlabel='Predicted class', ylabel='True class', title='Confusion matrix')
for row in range(num_classes):
    for col in range(num_classes):
        count = cm[row, col].item()
        color = 'white' if count > cm.max().item() / 2 else 'black'
        ax.text(col, row, count, ha='center', va='center', color=color, fontsize=8)
fig.colorbar(heatmap, ax=ax, label='Images')
fig.tight_layout()
plt.show()

errors_only = cm.clone()
errors_only.fill_diagonal_(0)
if errors_only.sum().item() > 0:
    row, col = divmod(errors_only.argmax().item(), num_classes)
    print(f'가장 많은 오분류 중 하나: {class_names[row]} → {class_names[col]} '
          f'({errors_only[row, col].item()}장)')

In [ ]:
error_indices = (true_labels != pred_labels).nonzero(as_tuple=True)[0]
if len(error_indices) == 0:
    print('오분류 이미지가 없습니다.')
else:
    count = min(6, len(error_indices))
    fig, axes = plt.subplots(2, 3, figsize=(11, 7))
    for ax in axes.flat:
        ax.axis('off')
    # 테스트 로더를 섞지 않았으므로, 예측 위치와 test_data의 인덱스가 일치
    for ax, index in zip(axes.flat, error_indices[:count].tolist()):
        image, actual = test_data[index]
        predicted = pred_labels[index].item()
        ax.imshow(image[0], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        ax.set_title(f'True: {class_names[actual.item()]}\n'
                     f'Pred: {class_names[predicted]} ({confidences[index].item():.0%})')
    fig.tight_layout()
    plt.show()

## 제출 답안 · 숫자만 적기

1. 두 번째 풀링 결과를 펼친 벡터의 길이는? ___개
2. 검증 손실이 가장 낮아서 선택된 모델은 몇 번째 epoch인가요? ___번째
3. 테스트 정확도는? ___%